# Notebook 14 — CARES-OOF-002: the redesigned proposed method (not baseline screening)

**Novelty is a research hypothesis, not a publication claim.** This experiment implements a *train-only cross-fitted multiclass correction* and a *train-only selected risk gate*. It is distinct from the earlier CARES-PAIR-001 shared-class pairwise experts, which decreased overall macro-F1. Its constituents (OOF stacking, calibration and selective classification) have extensive prior work; a literature audit is necessary before claiming a genuinely novel combination.

**Experimental design:** Derive two sets of OOF 10-class LightGBM distributions across three stratified folds from the designated **140,328 original training targets**, fitting six genuine fold models; a multiclass L2-regularized residual meta-model uses 75% of OOF training examples; the remaining 25% selects correction strength and confidence gate subject to development-training accuracy, Backdoor and Analysis constraints. Then apply the learned corrector, without fitting it on any original validation labels, to the **frozen pre-existing LightGBM predictions** on the **35,005 original internal-validation targets**. Four prespecified variants: unchanged baseline, simple averaging, unconstrained meta model, and risk-gated meta model. **The official 82,332-record test partition remains locked.**

**Reproducibility:** Run as Kaggle **Save Version → Save & Run All**. The clean cloud session restores both previous frozen SHA256-verified model archives from private GitHub draft releases. The notebook trains six folds once and archives all trained checkpoints, model probabilities, label-free routing configuration, all four true confusion matrices and per-class tables in an unpublished release, uploading public aggregate metrics to GitHub. Kaggle `pushRare` secret must be enabled for this new notebook. Internet ON; attach `dsshayan/unsw-nb15`. CPU is sufficient. No need to keep browser or computer on.

**Failure honesty:** If the method yields no satisfactory improvement, a training-only gate is allowed to fall back to the original baseline. This is a valuable negative experimental result; we must not force a higher metric through test or validation leakage.

In [ ]:
# ONE Kaggle cell: restore frozen baselines, train first redesigned CARES methodology,
# evaluate on original internal validation, independently audit and archive everything.
import os,subprocess,sys,importlib.util,hashlib
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO=Path("/kaggle/working/rare-attack-ids-cares-oof")
BASE=Path("/kaggle/working/research_outputs")
RUN=BASE/"CARES-OOF-002-S42"
CSV=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
assert CSV.is_file(), "Attach dataset dsshayan/unsw-nb15 first"
EXPECTED="bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
with CSV.open("rb") as file:
    assert hashlib.file_digest(file,"sha256").hexdigest()==EXPECTED, "Dataset checksum mismatch"
secret=UserSecretsClient().get_secret("pushRare")
assert secret, "Authorize Kaggle secret pushRare in THIS notebook"
if not REPO.exists():
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git",
        str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
if importlib.util.find_spec("lightgbm") is None:
    subprocess.run([sys.executable,"-m","pip","install","-q","lightgbm>=4.1,<5"],check=True)
env=os.environ.copy()
env["PYTHONPATH"]=str(REPO/"src")
env["GITHUB_TOKEN"]=secret
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"
print("CARES experiment repository:",subprocess.check_output(
    ["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip(),flush=True)

for script,archive in (
    ("restore_scv_release.py","SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar"),
    ("restore_ctrl003_tabular.py","CTRL-TAB-001-S42_FULL_ARTIFACTS.tar")
):
    subprocess.run([sys.executable,str(REPO/"scripts"/script),
       "--base",str(BASE),"--archive",str(Path("/kaggle/working")/archive)],
       cwd=str(REPO),env=env,check=True)

subprocess.run([
    sys.executable,str(REPO/"scripts/run_cares_oof002.py"),
    "--repo",str(REPO),
    "--reference",str(BASE/"SCV-IDS-REPRO-001"),
    "--baseline",str(BASE/"CTRL-TAB-001-S42"),
    "--config",str(REPO/"configs/cares_oof002_seed42.json"),
    "--output",str(RUN)
],cwd=str(REPO),env=env,check=True)
print("\nRESEARCH RESULTS:")
print((RUN/"comparison_scores.csv").read_text())
subprocess.run([
    sys.executable,str(REPO/"scripts/archive_sync_cares_oof002.py"),
    "--run",str(RUN),
    "--archive","/kaggle/working/CARES-OOF-002-S42_FULL_ARTIFACTS.tar"
],cwd=str(REPO),env=env,check=True)
print("CARES-OOF-002 PROPOSED METHODOLOGY VERIFIED AND ARCHIVED",flush=True)


After the cloud run, send back the lines **`CARES-OOF-002 PROPOSED METHODOLOGY COMPLETE`**, **`CARES-OOF PRIVATE DRAFT RELEASE VERIFIED`**, and **`CARES-OOF VERIFIED RESULTS COMMIT`**. A new CI passing synthetic checks confirms code integrity but cannot guarantee Kaggle runtime success or a higher score. This is our actual redesigned proposed-method experiment; CatBoost baseline benchmarking is complete.